# M3C2 reproducibility test

The same two epochs, core points, and parameters are processed repeatedly. Distance estimates and normal directions are compared between runs.

In [ ]:
import os
from datetime import datetime
from pathlib import Path
import time

import numpy as np
import pandas as pd
import py4dgeo

print("py4dgeo:", getattr(py4dgeo, "__version__", "unknown"))
print("py4dgeo location:", py4dgeo.__file__)

## Configuration

In [ ]:
DATA_PATH = Path(r"C:\rsa\research_proj\blender_project\simulation_test2\output\simulation_test2_als_downsampled1")
REFERENCE_FILE = DATA_PATH / "simulation_test20005.xyz"
TARGET_FILE = DATA_PATH / "simulation_test20021.xyz"
REFERENCE_TIMESTAMP = datetime(2020, 1, 6)
TARGET_TIMESTAMP = datetime(2020, 1, 22)

NORMAL_RADIUS = 0.5
CYLINDER_RADIUS = 1.0
MAX_DISTANCE = 10.0
REGISTRATION_ERROR = 0.01
N_REPETITIONS = 20

# 1=lower-left, 2=lower-right, 3=upper-left, 4=upper-right.
selected_block = 1
scene_x_range = (-20.0, 60.0)
scene_y_range = (-20.0, 60.0)

OUTPUT_DIR = Path.cwd() / f"m3c2_reproducibility_{selected_block}"
OUTPUT_DIR.mkdir(exist_ok=True)

## Input data and core points

In [ ]:
reference_epoch, target_epoch = py4dgeo.read_from_xyz(
    REFERENCE_FILE, TARGET_FILE
)
reference_epoch.timestamp = REFERENCE_TIMESTAMP
target_epoch.timestamp = TARGET_TIMESTAMP
all_corepoints = reference_epoch.cloud[::10]  # Use every 10th point as a core point

x_min, x_max = scene_x_range
y_min, y_max = scene_y_range
x_mid = 0.5 * (x_min + x_max)
y_mid = 0.5 * (y_min + y_max)

block_bounds = {
    1: (x_min, x_mid, y_min, y_mid),
    2: (x_mid, x_max, y_min, y_mid),
    3: (x_min, x_mid, y_mid, y_max),
    4: (x_mid, x_max, y_mid, y_max),
}

if selected_block == "all":
    corepoints = all_corepoints.copy()
    print(f"Selected full scene: x=[{x_min}, {x_max}], y=[{y_min}, {y_max}]")
else:
    if selected_block not in block_bounds:
        raise ValueError(
            f"selected_block must be 'all' or one of {sorted(block_bounds)}, "
            f"got {selected_block!r}"
        )

    block_x_min, block_x_max, block_y_min, block_y_max = block_bounds[selected_block]
    x_in_block = (all_corepoints[:, 0] >= block_x_min) & (
        all_corepoints[:, 0] < block_x_max
        if selected_block in (1, 3)
        else all_corepoints[:, 0] <= block_x_max
    )
    y_in_block = (all_corepoints[:, 1] >= block_y_min) & (
        all_corepoints[:, 1] < block_y_max
        if selected_block in (1, 2)
        else all_corepoints[:, 1] <= block_y_max
    )
    corepoints = all_corepoints[x_in_block & y_in_block].copy()

    if len(corepoints) == 0:
        raise ValueError(f"No core points found in block {selected_block}")

    print(
        f"Selected block {selected_block}: "
        f"x=[{block_x_min}, {block_x_max}], "
        f"y=[{block_y_min}, {block_y_max}]"
    )

print(f"Core points: {len(corepoints):,}/{len(all_corepoints):,}")

## Repeated calculations

In [ ]:
class RecordingM3C2(py4dgeo.M3C2):
    def directions(self):
        directions = np.asarray(super().directions(), dtype=float).copy()
        self.recorded_directions = directions
        return directions


def new_m3c2(algorithm_class=RecordingM3C2, **kwargs):
    return algorithm_class(
        epochs=(reference_epoch, target_epoch),
        corepoints=corepoints.copy(),
        normal_radii=[NORMAL_RADIUS],
        cyl_radius=CYLINDER_RADIUS,
        max_distance=MAX_DISTANCE,
        registration_error=REGISTRATION_ERROR,
        **kwargs,
    )


def run_direct():
    algorithm = new_m3c2()
    start = time.perf_counter()
    distances, _ = algorithm.run()
    elapsed = time.perf_counter() - start
    return {
        "distances": np.asarray(distances, dtype=float).copy(),
        "directions": algorithm.recorded_directions.copy(),
        "elapsed_s": elapsed,
    }


def run_archive(repetition):
    algorithm = new_m3c2()
    path = OUTPUT_DIR / f"m3c2_repetition_{repetition:02d}.zip"
    analysis = py4dgeo.SpatiotemporalAnalysis(str(path), force=True)
    analysis.reference_epoch = reference_epoch
    analysis.corepoints = corepoints.copy()
    analysis.m3c2 = algorithm

    start = time.perf_counter()
    analysis.add_epochs(target_epoch)
    elapsed = time.perf_counter() - start
    return {
        "distances": analysis.distances[:, 0].astype(float).copy(),
        "directions": algorithm.recorded_directions.copy(),
        "elapsed_s": elapsed,
    }


direct_runs = [run_direct() for _ in range(N_REPETITIONS)]
archive_runs = [run_archive(i) for i in range(1, N_REPETITIONS + 1)]

for label, runs in [("run()", direct_runs), ("add_epochs()", archive_runs)]:
    for repetition, result in enumerate(runs, start=1):
        print(
            f"{label} {repetition:02d}: "
            f"{np.isfinite(result['distances']).sum():,} valid, "
            f"{result['elapsed_s']:.3f} s"
        )

## Differences from the first run

In [ ]:
def compare_distances(candidate, baseline):
    valid_candidate = np.isfinite(candidate)
    valid_baseline = np.isfinite(baseline)
    common = valid_candidate & valid_baseline
    difference = candidate[common] - baseline[common]

    return {
        "valid_distances": int(valid_candidate.sum()),
        "changed_distance_validity": int(np.count_nonzero(valid_candidate != valid_baseline)),
        "distance_mae_m": np.mean(np.abs(difference)) if difference.size else np.nan,
        "distance_rmse_m": np.sqrt(np.mean(difference**2)) if difference.size else np.nan,
        "maximum_distance_difference_m": np.max(np.abs(difference)) if difference.size else np.nan,
        "distances_identical": np.array_equal(candidate, baseline, equal_nan=True),
    }


def compare_directions(candidate, baseline):
    candidate = np.asarray(candidate, dtype=float)
    baseline = np.asarray(baseline, dtype=float)

    valid_candidate = np.all(np.isfinite(candidate), axis=1)
    valid_baseline = np.all(np.isfinite(baseline), axis=1)
    common = valid_candidate & valid_baseline

    candidate_common = candidate[common]
    baseline_common = baseline[common]
    norm_candidate = np.linalg.norm(candidate_common, axis=1)
    norm_baseline = np.linalg.norm(baseline_common, axis=1)
    nonzero = (norm_candidate > 0) & (norm_baseline > 0)

    vector_difference = np.linalg.norm(
        candidate_common - baseline_common, axis=1
    )
    cosine = np.sum(
        candidate_common[nonzero] * baseline_common[nonzero], axis=1
    ) / (norm_candidate[nonzero] * norm_baseline[nonzero])
    angles = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

    return {
        "changed_direction_validity": int(np.count_nonzero(valid_candidate != valid_baseline)),
        "directions_changed": int(np.count_nonzero(vector_difference > 1e-12)),
        "mean_direction_angle_deg": np.mean(angles) if angles.size else np.nan,
        "maximum_direction_angle_deg": np.max(angles) if angles.size else np.nan,
        "maximum_direction_vector_difference": np.max(vector_difference) if vector_difference.size else np.nan,
        "directions_identical": np.array_equal(candidate, baseline, equal_nan=True),
    }


def compare_with_first(runs, method):
    rows = []
    baseline = runs[0]
    for repetition, result in enumerate(runs, start=1):
        rows.append({
            "method": method,
            "repetition": repetition,
            **compare_distances(result["distances"], baseline["distances"]),
            **compare_directions(result["directions"], baseline["directions"]),
        })
    return rows


within_method = pd.DataFrame(
    compare_with_first(direct_runs, "M3C2.run()")
    + compare_with_first(archive_runs, "add_epochs()")
)

display(within_method)
within_method.to_csv(OUTPUT_DIR / "distance_and_direction_comparison.csv", index=False)

## Comparison between execution methods

In [ ]:
between_methods = pd.DataFrame([
    {
        "repetition": repetition,
        **compare_distances(archive["distances"], direct["distances"]),
        **compare_directions(archive["directions"], direct["directions"]),
    }
    for repetition, (direct, archive) in enumerate(
        zip(direct_runs, archive_runs), start=1
    )
])

display(between_methods)
between_methods.to_csv(OUTPUT_DIR / "between_methods_comparison.csv", index=False)

## Fixed-direction test

The normal directions recorded in the first direct run are reused for all subsequent distance calculations.

In [ ]:
fixed_directions = direct_runs[0]["directions"].copy()


class FixedDirectionsM3C2(py4dgeo.M3C2):
    def __init__(self, *args, fixed_directions, **kwargs):
        super().__init__(*args, **kwargs)
        self.fixed_directions = np.asarray(fixed_directions, dtype=float).copy()

    def directions(self):
        return self.fixed_directions.copy()


def run_with_fixed_directions():
    algorithm = new_m3c2(
        FixedDirectionsM3C2,
        fixed_directions=fixed_directions,
    )
    distances, _ = algorithm.run()
    return np.asarray(distances, dtype=float).copy()


def run_archive_with_fixed_directions(repetition):
    algorithm = new_m3c2(
        FixedDirectionsM3C2,
        fixed_directions=fixed_directions,
    )
    path = OUTPUT_DIR / f"m3c2_fixed_repetition_{repetition:02d}.zip"
    analysis = py4dgeo.SpatiotemporalAnalysis(str(path), force=True)
    analysis.reference_epoch = reference_epoch
    analysis.corepoints = corepoints.copy()
    analysis.m3c2 = algorithm
    analysis.add_epochs(target_epoch)
    return analysis.distances[:, 0].astype(float).copy()


fixed_direct_runs = [
    run_with_fixed_directions() for _ in range(N_REPETITIONS)
]

fixed_archive_runs = [
    run_archive_with_fixed_directions(repetition)
    for repetition in range(1, N_REPETITIONS + 1)
]

fixed_direction_comparison = pd.DataFrame(
    [
        {
            "method": "M3C2.run() with fixed directions",
            "repetition": repetition,
            **compare_distances(distances, fixed_direct_runs[0]),
        }
        for repetition, distances in enumerate(fixed_direct_runs, start=1)
    ]
    + [
        {
            "method": "add_epochs() with fixed directions",
            "repetition": repetition,
            **compare_distances(distances, fixed_archive_runs[0]),
        }
        for repetition, distances in enumerate(fixed_archive_runs, start=1)
    ]
)

display(fixed_direction_comparison)
fixed_direction_comparison.to_csv(
    OUTPUT_DIR / "fixed_direction_comparison.csv", index=False
)

fixed_between_methods = pd.DataFrame([
    {
        "repetition": repetition,
        **compare_distances(archive, direct),
    }
    for repetition, (direct, archive) in enumerate(
        zip(fixed_direct_runs, fixed_archive_runs), start=1
    )
])

display(fixed_between_methods)
fixed_between_methods.to_csv(
    OUTPUT_DIR / "fixed_between_methods_comparison.csv", index=False
)

## Summary

In [ ]:
summary = []

for method, group in within_method.groupby("method", sort=False):
    repeated = group[group["repetition"] > 1]
    summary.append({
        "test": method,
        "identical_distances": int(repeated["distances_identical"].sum()),
        "different_distances": int((~repeated["distances_identical"]).sum()),
        "identical_directions": int(repeated["directions_identical"].sum()),
        "different_directions": int((~repeated["directions_identical"]).sum()),
        "maximum_distance_difference_m": repeated["maximum_distance_difference_m"].max(),
        "maximum_direction_angle_deg": repeated["maximum_direction_angle_deg"].max(),
    })

summary.append({
    "test": "run() versus add_epochs()",
    "identical_distances": int(between_methods["distances_identical"].sum()),
    "different_distances": int((~between_methods["distances_identical"]).sum()),
    "identical_directions": int(between_methods["directions_identical"].sum()),
    "different_directions": int((~between_methods["directions_identical"]).sum()),
    "maximum_distance_difference_m": between_methods["maximum_distance_difference_m"].max(),
    "maximum_direction_angle_deg": between_methods["maximum_direction_angle_deg"].max(),
})

for method, group in fixed_direction_comparison.groupby("method", sort=False):
    repeated = group[group["repetition"] > 1]
    summary.append({
        "test": method,
        "identical_distances": int(repeated["distances_identical"].sum()),
        "different_distances": int((~repeated["distances_identical"]).sum()),
        "identical_directions": N_REPETITIONS - 1,
        "different_directions": 0,
        "maximum_distance_difference_m": repeated["maximum_distance_difference_m"].max(),
        "maximum_direction_angle_deg": 0.0,
    })

summary.append({
    "test": "fixed run() versus fixed add_epochs()",
    "identical_distances": int(fixed_between_methods["distances_identical"].sum()),
    "different_distances": int((~fixed_between_methods["distances_identical"]).sum()),
    "identical_directions": N_REPETITIONS,
    "different_directions": 0,
    "maximum_distance_difference_m": fixed_between_methods["maximum_distance_difference_m"].max(),
    "maximum_direction_angle_deg": 0.0,
})

summary = pd.DataFrame(summary)
display(summary)
summary.to_csv(OUTPUT_DIR / "reproducibility_summary.csv", index=False)